In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Intrări și ieșiri ale primitivelor

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Versiunea beta a unui nou model de execuție este acum disponibilă. Modelul de execuție dirijată oferă mai multă flexibilitate atunci când îți personalizezi fluxul de lucru pentru atenuarea erorilor. Consultă ghidul [Modelul de execuție dirijată](/guides/directed-execution-model) pentru mai multe informații.


<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Această pagină oferă o prezentare generală a intrărilor și ieșirilor primitivelor Qiskit Runtime care execută sarcini de lucru pe resursele de calcul IBM Quantum&reg;. Aceste primitive îți oferă posibilitatea de a defini în mod eficient sarcini de lucru vectorizate folosind o structură de date cunoscută sub numele de **Primitive Unified Bloc (PUB)**. Aceste PUBuri reprezintă unitatea fundamentală de lucru de care are nevoie un QPU pentru a executa aceste sarcini de lucru. Ele sunt utilizate ca intrări pentru metoda [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) a primitivelor Sampler și Estimator, care execută sarcina de lucru definită ca un job. Apoi, după ce job-ul s-a finalizat, rezultatele sunt returnate într-un format care depinde atât de PUBurile utilizate, cât și de opțiunile de runtime specificate de primitivele Sampler sau Estimator.
<span id="pubs"></span>
## Prezentare generală a PUBurilor
Când apelezi metoda [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) a unei primitive, principalul argument necesar este o `list` cu unul sau mai multe tupluri — câte unul pentru fiecare Circuit executat de primitivă. Fiecare dintre aceste tupluri este considerat un PUB, iar elementele obligatorii ale fiecărui tuplu din listă depind de primitiva utilizată. Datele furnizate acestor tupluri pot fi, de asemenea, organizate într-o varietate de forme pentru a oferi flexibilitate în cadrul unei sarcini de lucru prin broadcasting — regulile căreia sunt descrise într-o [secțiune ulterioară](#broadcasting-rules).

### Estimator PUB
Pentru primitiva Estimator, formatul PUB ar trebui să conțină cel mult patru valori:
- Un singur `QuantumCircuit`, care poate conține unul sau mai multe obiecte [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- O listă cu unul sau mai multe observabile, care specifică valorile de așteptare de estimat, organizate într-un tablou (de exemplu, un singur observabil reprezentat ca un tablou 0-d, o listă de observabile ca un tablou 1-d și așa mai departe). Datele pot fi în oricare dintre formatele `ObservablesArrayLike`, cum ar fi `Pauli`, `SparsePauliOp`, `PauliList` sau `str`.
   > **Note:** Dacă ai două observabile care comută în PUBuri diferite, dar cu același Circuit, acestea nu vor fi estimate folosind aceeași măsurătoare. Fiecare PUB reprezintă o bază diferită pentru măsurătoare și, prin urmare, sunt necesare măsurători separate pentru fiecare PUB. Pentru a te asigura că observabilele care comută sunt estimate folosind aceeași măsurătoare, acestea trebuie grupate în același PUB.
- O colecție de valori ale parametrilor pentru a lega Circuit-ul. Aceasta poate fi specificată ca un singur obiect de tip tablou în care ultimul indice este peste obiectele `Parameter` ale Circuit-ului, sau omisă (sau echivalent, setată la `None`) dacă Circuit-ul nu are obiecte `Parameter`.
- (Opțional) o precizie țintă pentru valorile de așteptare de estimat

### Sampler PUB
Pentru primitiva Sampler, formatul tuplului PUB conține cel mult trei valori:
- Un singur `QuantumCircuit`, care poate conține unul sau mai multe obiecte [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
   *Notă: Aceste Circuit-uri ar trebui să includă și instrucțiuni de măsurare pentru fiecare dintre Qubit-urile care urmează să fie eșantionate.*
- O colecție de valori ale parametrilor pentru a lega Circuit-ul împotriva $\theta_k$ (necesară doar dacă sunt utilizate obiecte `Parameter` care trebuie legate la momentul execuției)
- (Opțional) un număr de shots cu care să se măsoare Circuit-ul
---

Următorul cod demonstrează un exemplu de set de intrări vectorizate pentru primitiva `Estimator` și le execută pe un Backend IBM&reg; ca un singur obiect `RuntimeJobV2 `.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV2 as Sampler,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

### Reguli de broadcasting
PUB-urile agregează elemente din mai multe array-uri (observabile și valori de parametri) urmând aceleași reguli de broadcasting ca NumPy. Această secțiune rezumă pe scurt acele reguli. Pentru o explicație detaliată, consultă [documentația NumPy privind regulile de broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Reguli:

* Array-urile de intrare nu trebuie să aibă același număr de dimensiuni.
  * Array-ul rezultat va avea același număr de dimensiuni ca array-ul de intrare cu cea mai mare dimensiune.
  * Dimensiunea fiecărei dimensiuni este cea mai mare dimensiune corespunzătoare.
  * Dimensiunile lipsă sunt considerate a avea dimensiunea unu.
* Comparațiile de formă încep cu dimensiunea cea mai din dreapta și continuă spre stânga.
* Două dimensiuni sunt compatibile dacă dimensiunile lor sunt egale sau dacă una dintre ele este 1.

Exemple de perechi de array-uri care fac broadcasting:

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

Exemple de perechi de array-uri care nu fac broadcasting:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

`EstimatorV2` returnează câte o estimare a valorii de așteptare pentru fiecare element al formei rezultate în urma broadcasting-ului.

Iată câteva exemple de tipare comune exprimate în termeni de broadcasting de array-uri. Reprezentarea lor vizuală corespunzătoare este prezentată în figura care urmează:

Seturile de valori de parametri sunt reprezentate prin array-uri n x m, iar array-urile de observabile sunt reprezentate prin unul sau mai multe array-uri cu o singură coloană. Pentru fiecare exemplu din codul anterior, seturile de valori de parametri sunt combinate cu array-ul lor de observabile pentru a crea estimările valorilor de așteptare rezultante.

 - *Exemplul 1*: (broadcasting al unui singur observabil) are un set de valori de parametri care este un array 5x1 și un array de observabile 1x1. Singurul element din array-ul de observabile este combinat cu fiecare element din setul de valori de parametri pentru a crea un singur array 5x1 în care fiecare element este o combinație a elementului original din setul de valori de parametri cu elementul din array-ul de observabile.

 - *Exemplul 2*: (zip) are un set de valori de parametri 5x1 și un array de observabile 5x1. Rezultatul este un array 5x1 în care fiecare element este o combinație a celui de-al n-lea element din setul de valori de parametri cu cel de-al n-lea element din array-ul de observabile.

  - *Exemplul 3*: (outer/produs) are un set de valori de parametri 1x6 și un array de observabile 4x1. Combinarea lor duce la un array 4x6 creat prin combinarea fiecărui element din setul de valori de parametri cu *fiecare* element din array-ul de observabile, astfel încât fiecare valoare de parametru devine o întreagă coloană în rezultat.

  - *Exemplul 4*: (generalizare standard nd) are un array de set de valori de parametri 3x6 și două array-uri de observabile 3x1. Acestea se combină pentru a crea două array-uri de ieșire 3x6 într-un mod similar cu exemplul anterior.

![Această imagine ilustrează mai multe reprezentări vizuale ale broadcasting-ului de array-uri](../docs/images/guides/primitive-input-output/broadcasting.svg "Reprezentare vizuală a broadcasting-ului")

In [4]:
# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=4096, num_bits=10>)

The shape of register `meas` is (4096, 2).

The bytes in register `alpha`, shot by shot:
[[  3 254]
 [  0   0]
 [  3 255]
 ...
 [  0   0]
 [  3 255]
 [  0   0]]



<Admonition type="tip" title="SparsePauliOp">
Fiecare `SparsePauliOp` contează ca un singur element în acest context, indiferent de numărul de Pauli conținuți în `SparsePauliOp`. Astfel, în scopul acestor reguli de broadcasting, toate elementele de mai jos au aceeași formă:

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'1111111110': 199, '0000000000': 1337, '1111111111': 1052, '1111111000': 33, '1110000000': 65, '1100100000': 2, '1100000000': 25, '0010001110': 1, '0000000011': 30, '1111111011': 58, '1111111010': 25, '0000000110': 7, '0010000001': 11, '0000000001': 179, '1110111110': 6, '1111110000': 33, '1111101111': 49, '1110111111': 40, '0000111010': 2, '0100000000': 35, '0000000010': 51, '0000100000': 31, '0110000000': 7, '0000001111': 22, '1111111100': 24, '1011111110': 5, '0001111111': 58, '0000111111': 24, '1111101110': 10, '0000010001': 5, '0000001001': 2, '0011111111': 38, '0000001000': 11, '1111100000': 34, '0111111111': 45, '0000000100': 18, '0000000101': 2, '1011111111': 11, '1110000001': 13, '1101111000': 1, '0010000000': 52, '0000010000': 17, '0000011111': 15, '1110100001': 1, '0111111110': 9, '0000000111': 19, '1101111111': 15, '1111110111': 17, '0011111110': 5, '0001101110': 1, '0111111011': 6, '0100001000': 2, '0010001111': 1, '1111011000': 1, '0000111110': 4, '0011110010': 1

Următoarele liste de operatori, deși echivalente în ceea ce privește informațiile conținute, au forme diferite:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=4096, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=4096, num_bits=9>)


</Admonition>
## Prezentare generală a rezultatelor primitive
Odată ce unul sau mai multe PUB-uri sunt trimise la un QPU pentru execuție și un job se finalizează cu succes, datele sunt returnate sub forma unui obiect container [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), accesat prin apelarea metodei `RuntimeJobV2.result()`. `PrimitiveResult` conține o listă iterabilă de obiecte [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) care conțin rezultatele execuției pentru fiecare PUB. În funcție de primitive-ul folosit, aceste date vor fi fie valori de așteptare și barele lor de eroare, în cazul Estimator-ului, fie eșantioane ale ieșirii Circuit-ului, în cazul Sampler-ului.

Fiecare element al acestei liste corespunde fiecărui PUB trimis la metoda `run()` a primitive-ului (de exemplu, un job trimis cu 20 de PUB-uri va returna un obiect `PrimitiveResult` care conține o listă cu 20 de `PubResult`-uri, câte unul corespunzător fiecărui PUB).

Fiecare dintre aceste obiecte `PubResult` posedă atât un atribut `data`, cât și un atribut `metadata`. Atributul `data` este un [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) personalizat, care conține valorile efective ale măsurătorilor, abaterile standard și alte informații similare. Acest `DataBin` are diverse atribute în funcție de forma sau structura PUB-ului asociat, precum și de opțiunile de mitigare a erorilor specificate de primitive-ul folosit pentru a trimite job-ul (de exemplu, [ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) sau [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)). Între timp, atributul `metadata` conține informații despre runtime și opțiunile de mitigare a erorilor utilizate (explicate mai târziu în secțiunea [Metadate ale rezultatelor](#result-metadata) de pe această pagină).

Urmează o schiță vizuală a structurii de date `PrimitiveResult`:

<Tabs>
    <TabItem value="estimator" label="Estimator Output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Sampler Output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second pub
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

Mai simplu spus, un singur job returnează un obiect `PrimitiveResult` și conține o listă cu unul sau mai multe obiecte `PubResult`. Aceste obiecte `PubResult` stochează apoi datele de măsurare pentru fiecare PUB care a fost trimis în job.

Fiecare `PubResult` posedă formate și atribute diferite în funcție de tipul de primitive folosit pentru job. Detaliile sunt explicate mai jos.

### Estimator output

Fiecare `PubResult` pentru primitive-ul Estimator conține cel puțin un tablou de valori de așteptare (`PubResult.data.evs`) și abateri standard asociate (fie `PubResult.data.stds`, fie `PubResult.data.ensemble_standard_error`, în funcție de `resilience_level` utilizat), dar poate conține mai multe date în funcție de opțiunile de mitigare a erorilor care au fost specificate.

Fragmentul de cod de mai jos descrie formatul `PrimitiveResult` (și al `PubResult`-ului asociat) pentru job-ul creat mai sus.

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (4096, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (4096, 2).
The bytes in register `beta`, shot by shot:
[[  0 135]
 [  0 247]
 [  1 247]
 ...
 [  0   0]
 [  1 224]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (4096, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  0 135]
 [  0 247]
 [  1 247]
 [  1 128]
 [  1 255]]

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.068359375
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.06396484375

The shape of the merged results is (4096, 2).
The bytes of the merged results:
[[  1  15]
 [  1 239]
 [  3 239]
 

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'execution' : {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 08:07:33', stop='2026-01-15 08:07:36', size=4096>)])},
'version' : 2,

The metadata of the PubResult result is:
'circuit_metadata' : {},


#### Cum calculează Estimator-ul eroarea
Pe lângă estimarea mediei observabilelor transmise în PUB-urile de intrare (câmpul `evs` din `DataBin`), Estimator-ul încearcă de asemenea să furnizeze o estimare a erorii asociate acelor valori de așteptare. Toate interogările Estimator-ului vor popula câmpul `stds` cu o cantitate similară erorii standard a mediei pentru fiecare valoare de așteptare, dar unele opțiuni de mitigare a erorilor produc informații suplimentare, cum ar fi `ensemble_standard_error`.

Consideră un singur observabil $\mathcal{O}$. În absența [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), poți gândi fiecare shot al execuției Estimator-ului ca furnizând o estimare punctuală a valorii de așteptare $\langle \mathcal{O} \rangle$. Dacă estimările punctuale se află într-un vector `Os`, atunci valoarea returnată în `ensemble_standard_error` este echivalentă cu următoarea (în care $\sigma_{\mathcal{O}}$ este [deviația standard a estimării valorii de așteptare](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) și $N_{shots}$ este numărul de shot-uri):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

care tratează toate shot-urile ca parte a unui singur ansamblu. Dacă ai solicitat [twirling](/guides/error-mitigation-and-suppression-techniques#pauli-twirling) pe Gate-uri (`twirling.enable_gates = True`), poți sorta estimările punctuale ale $\langle \mathcal{O} \rangle$ în seturi care împart un twirl comun. Numește aceste seturi de estimări `O_twirls`, și există `num_randomizations` (numărul de twirl-uri) dintre ele. Atunci `stds` este eroarea standard a mediei `O_twirls`, astfel:

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

unde $\sigma_{\mathcal{O}}$ este deviația standard a `O_twirls` și $N_{twirls}$ este numărul de twirl-uri. Când nu activezi twirling-ul, `stds` și `ensemble_standard_error` sunt egale.

Dacă activezi ZNE, atunci `stds` descrise mai sus devin ponderi într-o regresie neliniară la un model extrapolator. Ceea ce este returnat în final în `stds` în acest caz este incertitudinea modelului de potrivire evaluată la un factor de zgomot egal cu zero. Când există o potrivire slabă sau o incertitudine mare în potrivire, `stds` raportate pot deveni foarte mari. Când ZNE este activat, `pub_result.data.evs_noise_factors` și `pub_result.data.stds_noise_factors` sunt de asemenea populate, astfel încât să poți efectua propria extrapolație.
### Sampler output

Când un job Sampler se finalizează cu succes, obiectul [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) returnat conține o listă de obiecte [`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult), câte unul per PUB. Compartimentele de date ale acestor obiecte `SamplerPubResult` sunt obiecte de tip dicționar care conțin câte un `BitArray` per `ClassicalRegister` din Circuit.

Clasa `BitArray` este un container pentru date de shot ordonate. Mai în detaliu, stochează bitstring-urile eșantionate ca octeți într-un tablou bidimensional. Axa cea mai din stânga a acestui tablou parcurge shot-urile ordonate, în timp ce axa cea mai din dreapta parcurge octeții.

Ca prim exemplu, să analizăm următorul Circuit cu zece qubiți: